<a href="https://colab.research.google.com/github/nsp8/Machine-Learning-Resources/blob/notes/notebooks/rice_type_bin_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install opendatasets --quiet
import opendatasets as od

od.download("https://www.kaggle.com/datasets/mssmartypants/rice-type-classification")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: bluerev
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/mssmartypants/rice-type-classification


100%|██████████| 888k/888k [00:00<00:00, 966kB/s]

In [6]:
from dataclasses import dataclass
import torch
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [3]:
DEVICE = "gpu" if torch.cuda.is_available() else "cpu"

In [7]:
class CustomDataset(Dataset):
    def __init__(self, x, y):
        self.X = torch.tensor(x, dtype=torch.float32).to(DEVICE)
        self.y = torch.tensor(y, dtype=torch.float32).to(DEVICE)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, _index):
        return self.X[_index], self.y[_index]


@dataclass
class DataComponents:
    training_data: DataLoader
    validation_data: DataLoader
    testing_data: DataLoader


class DataPreprocessor:
    def __init__(self, file_data: pd.DataFrame):
        self.dataframe = file_data.copy(deep=True)

    def drop_nulls(self):
        self.dataframe.dropna(inplace=True)

    def reassign_id(self, id_column: str):
        self.dataframe.set_index(keys=[id_column], inplace=True)

    def get_unique_values(self, column: str):
        series = self.dataframe[column]
        print(series.value_counts())
        return series.unique()

    def normalize_data(self):
        for col in self.dataframe.columns:
            self.dataframe[col] = self.dataframe[col]/self.dataframe[col].abs().max()

    @property
    def X(self):
        return np.array(self.dataframe.iloc[:, :-1])

    @property
    def y(self):
        return np.array(self.dataframe.iloc[:, -1])

    def split_data(
        self,
        test_split: float = 0.3,
        val_split: float = 0.5,
        batch_size: int = 8,
        shuffle: bool = True
    ):
        x_train, x_test, y_train, y_test = train_test_split(self.X, self.y, test_size=test_split)
        x_test, x_val, y_test, y_val = train_test_split(x_test, y_test, test_size=val_split)
        return DataComponents(
            training_data=DataLoader(
                CustomDataset(x_train, y_train),
                batch_size=batch_size,
                shuffle=shuffle
            ),
            validation_data=DataLoader(
                CustomDataset(x_val, y_val),
                batch_size=batch_size,
                shuffle=shuffle
            ),
            testing_data=DataLoader(
                CustomDataset(x_test, y_test),
                batch_size=batch_size,
                shuffle=shuffle
            )
        )


In [8]:
class BinaryClassificationModel(torch.nn.Module):

    HIDDEN_NEURONS = 16

    def __init__(self, X):
        super().__init__()
        self.input_layer = torch.nn.Linear(X.shape[1], self.HIDDEN_NEURONS)
        self.linear = torch.nn.Linear(
            in_features=self.HIDDEN_NEURONS,
            out_features=1  # because it's binary classification
        )
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.input_layer(x)
        x = self.linear(x)
        x = self.sigmoid(x)
        return x
